# 🎙 EchoForge — Google Colab
**GPU:** T4 (15GB VRAM) &nbsp;|&nbsp; **Storage:** Google Drive

Tüm modeller ve çıktılar `GDrive/EchoForge/` altında saklanır.
Tarayıcı arayüzüne erişmek için son hücredeki **public URL**'i kullan.

In [ ]:
# ── Hücre 1: Google Drive Bağla ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive bağlandı')

In [ ]:
# ── Hücre 2: EchoForge Klonla / Güncelle ────────────────────────────────
import os

REPO_URL  = 'https://github.com/minniesmick/EchoForger.git'
REPO_PATH = '/content/drive/MyDrive/EchoForge/EchoForger'

if os.path.exists(REPO_PATH):
    %cd $REPO_PATH
    !git pull
else:
    !git clone $REPO_URL $REPO_PATH
    %cd $REPO_PATH

print('✅ Kod hazır')

In [ ]:
# ── Hücre 3: Bağımlılıkları Kur ─────────────────────────────────────────
# PyTorch Colab'da zaten kurulu (CUDA 12.x), sadece eksikler eklenir

!pip install -q \
    faster-whisper \
    TTS \
    pydub \
    soundfile \
    pyqtgraph \
    google-generativeai \
    gradio

print('✅ Bağımlılıklar kuruldu')

In [ ]:
# ── Hücre 4: Model Dizinini GDrive'a Bağla ──────────────────────────────
from pathlib import Path

GDRIVE_BASE   = Path('/content/drive/MyDrive/EchoForge')
MODEL_DIR     = GDRIVE_BASE / 'Ses_Modelleri'
HF_CACHE_DIR  = GDRIVE_BASE / 'hf_cache'
OUTPUT_DIR    = GDRIVE_BASE / 'output'

for d in [MODEL_DIR, HF_CACHE_DIR, OUTPUT_DIR,
          OUTPUT_DIR / 'transcripts', OUTPUT_DIR / 'audio']:
    d.mkdir(parents=True, exist_ok=True)

import os
os.environ['HF_HOME']      = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['TTS_HOME']     = str(MODEL_DIR)

print(f'✅ Klasörler hazır: {GDRIVE_BASE}')

In [ ]:
# ── Hücre 5: API Anahtarları ─────────────────────────────────────────────
# Colab Sol Panel → 🔑 Secrets → GEMINI_API_KEY ekle
# Buraya yazmana gerek yok, otomatik okunur.

from google.colab import userdata
import os

try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✅ Gemini API key yüklendi')
except Exception:
    print('⚠️  GEMINI_API_KEY bulunamadı — Ollama kullanılacak')

In [ ]:
# ── Hücre 6: GPU Kontrolü ────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU bulunamadı — CPU modunda çalışılacak')

In [ ]:
# ── Hücre 7: EchoForge Gradio Arayüzünü Başlat ──────────────────────────
# Bu hücreyi çalıştır ve çıkan public URL'i tarayıcıda aç

import sys
sys.argv = ['main.py', '--colab']  # Colab modunu aktifleştir

from colab_ui import launch_gradio
launch_gradio(share=True)          # share=True → MacBook'tan erişim için public URL